# 01 — Exploración de Datos de Sensores

Este notebook consulta InfluxDB para explorar las lecturas de sensores IoT
y visualizar series temporales con Plotly.

**Prerequisitos:** El pipeline debe estar corriendo:
```bash
python src/01_ingestion/sensor_simulator.py --devices 3 --interval 2
python src/01_ingestion/mqtt_to_kafka.py
python src/03_storage/kafka_to_influx.py
```

In [ ]:
import os
import pandas as pd
import plotly.express as px
from influxdb_client import InfluxDBClient

INFLUX_URL    = os.getenv('INFLUX_URL',    'http://localhost:18086')
INFLUX_TOKEN  = os.getenv('INFLUX_TOKEN',  'supersecrettoken')
INFLUX_ORG    = os.getenv('INFLUX_ORG',    'ilerna')
INFLUX_BUCKET = os.getenv('INFLUX_BUCKET', 'sensores')

client = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG)
query_api = client.query_api()
print('Conectado a InfluxDB:', INFLUX_URL)

In [ ]:
# ── Cargar últimos 30 minutos de datos ──────────────────────
RANGE = '30m'

query = f'''
from(bucket: "{INFLUX_BUCKET}")
  |> range(start: -{RANGE})
  |> filter(fn: (r) => r._measurement == "sensor_reading")
  |> pivot(rowKey: ["_time"], columnKey: ["_field"], valueColumn: "_value")
'''

df = query_api.query_data_frame(query, org=INFLUX_ORG)
print(f'Registros cargados: {len(df)}')
df.head()

In [ ]:
# ── Limpiar y preparar el DataFrame ─────────────────────────
if not df.empty:
    keep = ['_time', 'device_id', 'sensor', 'value', 'unit']
    cols = [c for c in keep if c in df.columns]
    df = df[cols].rename(columns={'_time': 'ts'})
    df['ts'] = pd.to_datetime(df['ts'], utc=True)
    df = df.sort_values('ts')
    print(df.dtypes)
    df.describe()

In [ ]:
# ── Serie temporal por tipo de sensor ───────────────────────
for sensor in df['sensor'].unique():
    sub = df[df['sensor'] == sensor]
    unit = sub['unit'].iloc[0] if 'unit' in sub.columns else ''
    fig = px.line(
        sub, x='ts', y='value', color='device_id',
        title=f'{sensor.capitalize()} ({unit})',
        labels={'ts': 'Tiempo', 'value': unit},
        template='plotly_dark',
    )
    fig.show()

In [ ]:
# ── Estadísticas descriptivas por (device, sensor) ──────────
stats = (
    df.groupby(['device_id', 'sensor'])['value']
    .agg(['count', 'mean', 'std', 'min', 'max'])
    .round(2)
)
stats

In [ ]:
# ── Distribución de valores por sensor ──────────────────────
fig = px.box(
    df, x='sensor', y='value', color='sensor',
    title='Distribución de valores por sensor',
    template='plotly_dark',
)
fig.show()

In [ ]:
client.close()
print('Conexión InfluxDB cerrada.')